# Molab: Chapter 4 experiments

Set REPO_DIR to a persistent directory. Run cells in order: setup, data preparation, calibration, training, isolated timing, reports. See docs/molab.md.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
repo = Path(os.environ.get('REPO_DIR', str(Path.cwd() / 'drl-context-poi'))).resolve()
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/sepehrmustafavi/drl-context-poi.git', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo) + '[dev,plots,baselines]'], check=True)
subprocess.run([sys.executable, 'scripts/check_environment.py'], cwd=repo, check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a CUDA runtime before continuing.')
for model in ('sasrec', 'getnext', 'dchl', 'ipcm'):
    subprocess.run([sys.executable, 'scripts/fetch_upstream.py', model], cwd=repo, check=True)
os.environ['DRLPOI_UPSTREAM_ROOT'] = str(repo / 'artifacts/upstream')
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=repo, check=True)


## Data and validation selection

Prepare data/canonical/{nyc,tky,gowalla_ca}.csv using docs/data.md and README.md before running this cell. Run calibration on the idle device. Existing frozen configurations are preserved.

In [ ]:
for dataset in ('nyc','tky','gowalla_ca'):
    csv = repo / 'data/canonical' / (dataset + '.csv')
    if not csv.exists():
        raise FileNotFoundError(csv)
    frozen = repo / 'configs/frozen' / (dataset + '.yaml')
    if not frozen.exists():
        config = 'configs/gowalla.yaml' if dataset == 'gowalla_ca' else 'configs/base.yaml'
        subprocess.run([sys.executable, '-m', 'drlpoi.cli', 'calibrate', '--csv', str(csv),
                        '--config', config, '--out', 'selection/' + dataset, '--export', str(frozen)],
                       cwd=repo, env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'), check=True)


## Training and evaluation

This cell waits for the worker to finish. Rerun it with the same plan and artifacts to resume. Keep other device workloads stopped before timing.

In [ ]:
plan = repo / 'experiments/final/plan.json'
if not plan.exists():
    subprocess.run([sys.executable, '-m', 'drlpoi.cli', 'plan', '--suite', 'full', '--settings-dir', 'configs/frozen', '--out', 'experiments/final'], cwd=repo, check=True)
subprocess.run([sys.executable, '-m', 'drlpoi.cli', 'worker', '--plan', str(plan), '--gpu', '0'], cwd=repo, check=True)


## Reference timing

Run only after all training workers have exited successfully and all other GPU jobs are stopped. The worker enforces completion of the non-timing phase. Three timing repeats use one checkpoint; they are not training seeds.

In [ ]:
subprocess.run([sys.executable, '-m', 'drlpoi.cli', 'worker', '--plan', str(plan), '--gpu', '0', '--timing'], cwd=repo, check=True)


## Reports

All figures/statistics below use completed real runs. Do not fill missing entries with synthetic test outputs. Report single-seed supplementary analyses and baseline adaptations explicitly.

In [ ]:
subprocess.run([sys.executable, '-m', 'drlpoi.cli', 'report', '--root', 'runs', '--out', 'artifacts/report'], cwd=repo, check=True)
for dataset in ('nyc','tky','gowalla_ca'):
    subprocess.run([sys.executable, '-m', 'drlpoi.cli', 'compare-family', '--family',
                    'experiments/final/family_' + dataset + '.json', '--out',
                    'artifacts/report/paired_' + dataset + '.json'], cwd=repo, check=True)
subprocess.run([sys.executable, '-m', 'drlpoi.cli', 'diagnostics', '--root', 'runs',
                '--out', 'artifacts/report/discussions.json'], cwd=repo, check=True)
subprocess.run([sys.executable, '-m', 'drlpoi.cli', 'figures', '--aggregate', 'artifacts/report/aggregate.json', '--out', 'artifacts/figures'], cwd=repo, check=True)
